# CSP — Problema de Satisfação de Restrições

**Inteligência Artificial 2026/27 — Grupo G05**

Neste notebook o problema de recolha e entrega é resolvido como um **CSP** (*Constraint Satisfaction Problem*).

Ao contrário do Greedy e do A\*, que constroem a rota passo a passo, o CSP **distribui as tarefas pelos veículos e pelas viagens** até tudo encaixar sem violar nenhuma regra. O objetivo é encontrar uma **solução viável**: os km não são o objetivo principal, entram como **restrição soft** (preferência).

O código está em `src/csp.py`.

## 1. Preparação

In [1]:
from pathlib import Path
import sys
import pandas as pd

# Pasta principal do projeto (funciona a partir da raiz ou da pasta notebooks/)
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from src.model import Problem, check_solution, print_solution
from src.csp import CSP, ac3, resolver_csp
from src.informed import greedy_search, heuristica1, heuristica2

problemas = {nome: Problem.from_csv(RAIZ / nome) for nome in ["data_tiny", "data_small"]}
for nome, p in problemas.items():
    print(f"{nome}: {len(p.orders)} encomendas, {len(p.requests)} itens, {len(p.vehicles)} veículos")

data_tiny: 12 encomendas, 40 itens, 2 veículos
data_small: 25 encomendas, 80 itens, 2 veículos


## 2. Formulação do problema

### Ideia: viagens em sequência
Cada veículo faz **viagens umas a seguir às outras** (1.ª, 2.ª, 3.ª…). Em cada viagem a ordem é sempre:

> produtores (recolher) → armazém (descarregar e carregar) → clientes (entregar)

Não se marcam horas: cada viagem começa logo que a anterior acaba, e as horas são calculadas pela simulação do `model.py` (as mesmas que o `check_solution` calcula). O CSP decide **que tarefas vão em cada viagem de cada veículo**.

### Variáveis
- `("R", i)` — **recolha** do item *i* (40 no tiny, 80 no small);
- `("E", o)` — **entrega** da encomenda *o* (12 no tiny, 25 no small).

A recolha e a entrega são variáveis diferentes porque muitos itens são recolhidos num dia e entregues no outro (ficam no armazém).

### Domínio
Pares **(veículo, nº da viagem)**, com até 10 viagens por veículo. Exemplo: `("R", 5) = (1, 0)` quer dizer *"o item 5 é recolhido pelo veículo 1 na 1.ª viagem"*.

### Restrições (hard)
1. **Capacidade da tarefa** (unária): a tarefa, sozinha, cabe no veículo.
2. **Capacidade da viagem**: o que se recolhe numa viagem cabe no veículo, e o que se carrega também.
3. **Precedência** (binária, entre a recolha de um item e a entrega da sua encomenda): no mesmo veículo, a recolha é na mesma viagem ou numa viagem anterior.
4. **Janelas de tempo** (recolha e entrega), **máximo de 24h no armazém**, **horizonte de 2 dias** e **espera entre veículos** (um veículo pode ter de esperar que o outro descarregue): verificadas como uma **restrição global**, simulando o plano com o `model.py`.

### Restrição soft
- **Menos km**: em cada variável experimentam-se primeiro os valores que dão menos km no total.

### Porque "nº da viagem" e não "hora de partida"?
A primeira versão usava como valor *(veículo, hora de partida)*. Não funcionou: o `model.py` não tem uma operação "esperar", por isso, se uma viagem anterior acabasse cedo, a simulação partia logo e fazia as recolhas horas antes do planeado. O plano deixava de bater certo com a execução (sobretudo nas 24h no armazém) e o problema ficava sem solução. Com viagens em sequência, o plano e a execução são iguais.

In [2]:
p = problemas["data_tiny"]
csp = CSP(p)

print("Número de variáveis:", len(csp.variaveis))
print("Exemplos:", csp.variaveis[:3], "...", csp.variaveis[-2:])

X = ("R", 1)
print(f"\nDomínio de {X} ({len(csp.dominios[X])} valores):")
print(csp.dominios[X])

print(f"\nVizinhos de {X} (precedência):", csp.vizinhos[X])
print(f"Vizinhos de ('E', 1):", sorted(csp.vizinhos[("E", 1)]))

Número de variáveis: 52
Exemplos: [('R', 1), ('R', 2), ('R', 3)] ... [('E', 11), ('E', 12)]

Domínio de ('R', 1) (20 valores):
[(1, 0), (1, 1), (1, 2), (1, 3), (1, 4), (1, 5), (1, 6), (1, 7), (1, 8), (1, 9), (2, 0), (2, 1), (2, 2), (2, 3), (2, 4), (2, 5), (2, 6), (2, 7), (2, 8), (2, 9)]

Vizinhos de ('R', 1) (precedência): {('E', 1)}
Vizinhos de ('E', 1): [('R', 1), ('R', 2), ('R', 3), ('R', 4)]


### Exemplo: uma viagem e as restrições
Uma viagem em que o veículo recolhe os itens da encomenda 1 e a entrega logo a seguir:

In [3]:
tarefas = [("R", i) for i in p.orders[1]] + [("E", 1)]
for operacao, item in csp.operacoes_viagem(tarefas):
    print(f"{operacao:<8} item {item:<3} em {p.name(p.place_of(operacao, p.request[item]))}")

pickup   item 3   em Horta Morais
pickup   item 4   em Horta Morais
pickup   item 1   em Campos Leite
pickup   item 2   em Hortas Valente
unload   item 1   em Centro de Recolha 1
unload   item 2   em Centro de Recolha 1
unload   item 3   em Centro de Recolha 1
unload   item 4   em Centro de Recolha 1
load     item 1   em Centro de Recolha 1
load     item 2   em Centro de Recolha 1
load     item 3   em Centro de Recolha 1
load     item 4   em Centro de Recolha 1
deliver  item 1   em Casa 3
deliver  item 2   em Casa 3
deliver  item 3   em Casa 3
deliver  item 4   em Casa 3


In [4]:
# Restrição 3 (precedência): no mesmo veículo, a recolha não pode ser depois da entrega
print("Recolha na 2.ª viagem e entrega na 1.ª (mesmo veículo):",
      csp.precedencia(("R", 1), (1, 1), ("E", 1), (1, 0)))
print("Recolha e entrega na mesma viagem:",
      csp.precedencia(("R", 1), (1, 0), ("E", 1), (1, 0)))

# Restrição 4 (global): simular o plano com o model.py
plano = {X: (1, 0) for X in tarefas}
print("\nKm do plano só com a encomenda 1 (veículo 1, 1.ª viagem):", csp.simular(plano))

Recolha na 2.ª viagem e entrega na 1.ª (mesmo veículo): False
Recolha e entrega na mesma viagem: True

Km do plano só com a encomenda 1 (veículo 1, 1.ª viagem): 156.36


## 3. Heurísticas

**Escolha da variável (que tarefa decidir a seguir):**
- **Urgência** (por omissão): a tarefa cuja janela fecha mais cedo. O plano é construído "pela ordem do relógio", o que evita conflitos de tempo.
- **MRV** (para comparação): a variável com menos valores possíveis, com desempate pela urgência e pelo grau (mais restrições com variáveis por atribuir).

**Escolha do valor (restrição soft):**
- **Menos km primeiro**: para cada valor possível simula-se o plano e experimentam-se primeiro os que dão menos km. Na prática, junta tarefas em viagens que já vão ao mesmo produtor ou cliente.

## 4. Algoritmos

- **AC-3** (antes da pesquisa): para cada par de variáveis vizinhas, retira valores que não têm nenhum valor compatível na outra variável.
- **Backtracking**: atribui uma variável de cada vez; se nenhum valor serve, **volta atrás** (backtrack) e muda a decisão anterior. O número de backtracks é contado.
- **Forward checking**: depois de cada atribuição, retira dos domínios das variáveis por atribuir os valores que deixaram de cumprir as restrições de capacidade e de precedência. Se algum domínio ficar vazio, volta atrás logo.

### AC-3 nos dois datasets

In [5]:
for nome, prob in problemas.items():
    c = CSP(prob)
    dominios = {X: list(D) for X, D in c.dominios.items()}
    antes = sum(len(D) for D in dominios.values())
    consistente, removidos = ac3(c, dominios)
    print(f"{nome}: {antes} valores antes do AC-3, {removidos} removidos (consistente: {consistente})")

data_tiny: 1040 valores antes do AC-3, 0 removidos (consistente: True)
data_small: 2100 valores antes do AC-3, 0 removidos (consistente: True)


O AC-3 **não remove nenhum valor** nestes dados. As restrições binárias (capacidade e precedência no mesmo veículo) quase nunca eliminam um valor sozinhas: há sempre outra viagem ou outro veículo possível. As restrições que realmente falham (janelas de tempo e 24h) dependem das horas, e essas só se conhecem simulando o plano completo — o AC-3 não as consegue ver.

## 5. Resolver: atribuição viável e número de backtracks

In [6]:
resultados = {}
linhas = []
for nome, prob in problemas.items():
    rotas, atribuicao, estat = resolver_csp(prob)
    valida, km, mensagem = check_solution(prob, rotas)
    resultados[nome] = (rotas, atribuicao)
    linhas.append({"Dataset": nome, "Solução válida": valida, "Distância (km)": km,
                   "Atribuições": estat["atribuicoes"], "Backtracks": estat["backtracks"],
                   "Tempo (s)": estat["segundos"]})
pd.DataFrame(linhas)

,Dataset,Solução válida,Distância (km),Atribuições,Backtracks,Tempo (s)
0,data_tiny,True,664.25,75,23,0.09
1,data_small,True,877.63,324,219,0.78


### A atribuição encontrada (tiny)
Cada linha é uma viagem: que itens recolhe e que encomendas entrega.

In [7]:
p = problemas["data_tiny"]
rotas, atribuicao = resultados["data_tiny"]

viagens = {}
for X, valor in atribuicao.items():
    viagens.setdefault(valor, []).append(X)

tabela = []
for (veiculo, k), tarefas in sorted(viagens.items()):
    tabela.append({"Veículo": veiculo, "Viagem": k + 1,
                   "Itens recolhidos": sorted(i for t, i in tarefas if t == "R"),
                   "Encomendas entregues": sorted(o for t, o in tarefas if t == "E")})
pd.DataFrame(tabela)

,Veículo,Viagem,Itens recolhidos,Encomendas entregues
0,1,1,"[5, 6, 11, 12, 13, 22, 23, 24, 25, 26, 27, 28,...","[4, 10, 12]"
1,1,2,"[1, 2, 3, 4, 7, 8, 9, 10, 14, 15, 16, 17, 18, ...","[1, 3, 5, 6, 7]"
2,2,1,[36],"[2, 8, 9, 11]"


### O plano de cada veículo (tiny)

In [8]:
print_solution(p, rotas)

Solução válida | total: 664.25 km

Veículo 1 (capacidade 150 kg)
  30/10 08:00  pickup   Horta Morais                 [6, 22, 26, 11, 24, 29, 35] +29.1 km
  30/10 09:30  pickup   Hortas Valente               [31, 32, 34, 5, 25, 40, 13, 37] +39.3 km
  30/10 10:11  pickup   Campos Leite                 [30, 33, 12, 23, 27, 28, 38, 39] +23.6 km
  30/10 11:11  unload   Centro de Recolha 1          [5, 6, 11, 12, 13, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 37, 38, 39, 40] +37.7 km
  30/10 11:26  load     Centro de Recolha 1          [11, 12, 13, 30, 31, 36, 37, 38, 39, 40] +0.0 km
  30/10 12:13  deliver  Casa 3                       [11, 12, 13]             +23.5 km
  30/10 12:56  deliver  Casa 4                       [30, 31]                 +25.2 km
  30/10 13:55  deliver  Casa 2                       [36, 37, 38, 39, 40]     +36.3 km
  31/10 07:45  pickup   Horta Morais                 [8, 3, 9, 14, 17, 4]     +52.9 km
  31/10 08:29  pickup   Campos Leite                 

## 6. Comparação das técnicas

Corre-se o CSP com variantes, para ver o efeito de cada técnica (limite de 20 segundos por execução).

In [9]:
variantes = {
    "Urgência + FC + soft (por omissão)": {},
    "Sem forward checking": {"usar_fc": False},
    "Sem restrição soft": {"soft": False},
    "MRV em vez de urgência": {"ordem": "mrv"},
    "Sem heurística (ordem da lista)": {"ordem": "lista"},
}
linhas = []
for nome, prob in problemas.items():
    for variante, opcoes in variantes.items():
        rotas_v, _, estat = resolver_csp(prob, max_segundos=20, **opcoes)
        linhas.append({"Dataset": nome, "Variante": variante,
                       "Distância (km)": check_solution(prob, rotas_v)[1] if rotas_v else "sem solução",
                       "Atribuições": estat["atribuicoes"], "Backtracks": estat["backtracks"],
                       "Tempo (s)": estat["segundos"]})
comparacao = pd.DataFrame(linhas)
comparacao

,Dataset,Variante,Distância (km),Atribuições,Backtracks,Tempo (s)
0,data_tiny,Urgência + FC + soft (por omissão),664.25,75,23,0.14
1,data_tiny,Sem forward checking,700.22,52,0,0.09
2,data_tiny,Sem restrição soft,664.25,80,28,0.10
3,data_tiny,MRV em vez de urgência,sem solução,10024,10024,20.00
4,data_tiny,Sem heurística (ordem da lista),sem solução,15668,15668,20.00
5,data_small,Urgência + FC + soft (por omissão),877.63,324,219,0.69
6,data_small,Sem forward checking,891.19,311,206,0.74
7,data_small,Sem restrição soft,951.63,345,240,1.04
8,data_small,MRV em vez de urgência,sem solução,7776,7776,20.01
9,data_small,Sem heurística (ordem da lista),sem solução,7852,7852,20.00


### Comparação com o Greedy
O CSP procura uma solução viável e o Greedy também; os dois cumprem todas as restrições, por isso os km são comparáveis.

In [10]:
linhas = []
for nome, prob in problemas.items():
    for h_nome, h in [("H1", heuristica1), ("H2", heuristica2)]:
        sol, _ = greedy_search(prob, prob.initial_state(), h)
        linhas.append({"Dataset": nome, "Algoritmo": f"Greedy {h_nome}",
                       "Distância (km)": check_solution(prob, sol)[1]})
    linhas.append({"Dataset": nome, "Algoritmo": "CSP",
                   "Distância (km)": check_solution(prob, resultados[nome][0])[1]})
pd.DataFrame(linhas)

,Dataset,Algoritmo,Distância (km)
0,data_tiny,Greedy H1,798.54
1,data_tiny,Greedy H2,802.24
2,data_tiny,CSP,664.25
3,data_small,Greedy H1,1070.60
4,data_small,Greedy H2,1009.19
5,data_small,CSP,877.63


## 7. Análise dos resultados

**Atribuição viável e backtracks.** O CSP encontra uma atribuição viável nos dois datasets: no tiny com **23 backtracks** e no small com **219 backtracks**, em menos de 1 segundo. O `check_solution` confirma que as soluções cumprem **todas** as restrições do problema (janelas de tempo, capacidade, 24h no armazém e horizonte), sem nenhuma relaxação.

**A heurística de escolha da variável é decisiva.**
- Com a **urgência** (a tarefa cuja janela fecha mais cedo) o plano é construído pela ordem do relógio, e há poucos conflitos.
- Com a **MRV** ou **sem heurística** (ordem da lista), o CSP não encontra solução em 20 segundos (cerca de 8 000 a 15 000 backtracks). Com o forward checking, os domínios ficam com tamanhos diferentes por causa da capacidade, e a MRV passa a escolher tarefas espalhadas no tempo; os conflitos de horas só aparecem muito mais tarde, o que obriga a voltar muito atrás.

**A restrição soft (menos km) funciona.** No small, a solução tem 877,63 km com a restrição soft e 951,63 km sem ela (74 km a menos). No tiny o resultado é igual.

**O forward checking e o AC-3 ajudam pouco.** O AC-3 não remove nenhum valor e o forward checking quase não muda o número de backtracks (no tiny até dá 0 backtracks sem ele, mas com mais km: 700,22 contra 664,25). Os dois só conseguem ver as restrições de capacidade e de precedência, que raramente falham. As restrições que falham (janelas e 24h) dependem das horas, e só a simulação do plano as consegue verificar.

**Comparação com o Greedy.** Embora o objetivo do CSP seja só encontrar uma solução viável, a restrição soft leva a soluções com menos km do que o Greedy: 664,25 contra 798,54 km no tiny e 877,63 contra 1009,19 km no small.

**Observação.** No tiny, o veículo 2 carrega encomendas no dia 30 e entrega parte delas no dia 31. As regras permitem-no (o limite de 24h conta só no armazém), mas na prática os produtos ficam muito tempo dentro do veículo.

## 8. Conclusão

O problema foi formulado como um CSP com uma variável por recolha de item e por entrega de encomenda, e com valores (veículo, nº da viagem). A decisão mais importante foi a forma de representar o tempo: com horas marcadas no domínio, o plano não batia certo com a execução e o problema ficava sem solução; com **viagens em sequência**, o plano e a execução coincidem e o CSP resolve os dois datasets com todas as restrições.

**Melhorias futuras:**
- Representar as janelas de tempo como restrições explícitas (por exemplo, cortar domínios com estimativas de horas), para que o AC-3 e o forward checking passem a ser úteis.
- Experimentar várias ordens de visita aos produtores em cada viagem (por exemplo, o mais próximo primeiro), para baixar os km.
- Uma MRV com domínios calculados por simulação (mais cara, mas mais informada).
- Limitar o tempo que os produtos podem ficar dentro do veículo.
